## Setup — configuration

In [ ]:
import os

# ---- knobs ----
N_ARTIFACTS = 10    # reproducible-pool artifacts
FLAKY_COUNT = 3     # flaky-pool artifacts
WORKDIR = "/content/bugswarm-pilot"

os.makedirs(WORKDIR, exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "logs"), exist_ok=True)
os.makedirs(os.path.join(WORKDIR, "parsed"), exist_ok=True)
os.chdir(WORKDIR)
print("working directory:", os.getcwd())
print(f"plan: {N_ARTIFACTS} reproducible + {FLAKY_COUNT} flaky-pool artifacts")


working directory: /content/bugswarm-pilot
plan: 10 reproducible + 3 flaky-pool artifacts


## Setup — install dependencies

In [ ]:
!pip install -q bugswarm-common

## Phase 1 — Query the BugSwarm API
Pulls the full candidate pools (active Python artifacts) and caches them locally. Retries with backoff on transient failures.

**Checkpoint:** expect ~1,900 reproducible + ~45 flaky artifacts cached to JSON.

In [ ]:
"""Phase 1 (robust): Query BugSwarm API with retries, backoff, incremental saves."""
import json
import time
import requests
from bugswarm.common.rest_api.database_api import DatabaseAPI

api = DatabaseAPI()

def q_with_retry(name, query_dict, max_tries=4):
    for attempt in range(1, max_tries + 1):
        try:
            print(f"Querying: {name} (attempt {attempt}) ...", flush=True)
            resp = api.filter_artifacts(json.dumps(query_dict))
            arts = resp if isinstance(resp, list) else resp.json()
            print(f"  -> {len(arts)} artifacts", flush=True)
            with open(f"artifacts_{name}.json", "w") as f:
                json.dump(arts, f)
            return arts
        except (requests.exceptions.RequestException, ConnectionError) as e:
            wait = 10 * attempt
            print(f"  !! {type(e).__name__}: {e}. Retrying in {wait}s ...", flush=True)
            time.sleep(wait)
    print(f"  XX {name} FAILED after {max_tries} tries", flush=True)
    return []

repro = q_with_retry("reproducible_py", {
    "status": "active",
    "lang": "Python",
    "reproducibility_status.status": "Reproducible",
})
time.sleep(5)
flaky = q_with_retry("flaky_py", {
    "status": "active",
    "lang": "Python",
    "reproducibility_status.status": "Flaky",
})

def summarize(arts, n=6):
    for a in arts[:n]:
        fj = a.get("failed_job", {}) or {}
        print("  -", a.get("image_tag"), "| repo:", a.get("repo"),
              "| job:", fj.get("job_id"))

print("\nSample reproducible:"); summarize(repro)
print("\nSample flaky:"); summarize(flaky)
print("\nDone.")


Querying: reproducible_py (attempt 1) ...
  -> 1919 artifacts
Querying: flaky_py (attempt 1) ...
  -> 45 artifacts

Sample reproducible:
  - bwhmather-verktyg-102468155 | repo: bwhmather/verktyg | job: 102468155
  - bwhmather-verktyg-102468154 | repo: bwhmather/verktyg | job: 102468154
  - bwhmather-verktyg-78833091 | repo: bwhmather/verktyg | job: 78833091
  - bwhmather-verktyg-109227527 | repo: bwhmather/verktyg | job: 109227527
  - bwhmather-verktyg-109183187 | repo: bwhmather/verktyg | job: 109183187
  - bwhmather-verktyg-109227526 | repo: bwhmather/verktyg | job: 109227526

Sample flaky:
  - numpy-numpy-105433442 | repo: numpy/numpy | job: 105433442
  - numpy-numpy-105433445 | repo: numpy/numpy | job: 105433445
  - numpy-numpy-106081978 | repo: numpy/numpy | job: 106081978
  - numpy-numpy-107623799 | repo: numpy/numpy | job: 107623799
  - numpy-numpy-135521497 | repo: numpy/numpy | job: 135521497
  - numpy-numpy-308527199 | repo: numpy/numpy | job: 308527199

Done.


## Phase 2 — Select artifacts
Picks `N_ARTIFACTS` reproducible + `FLAKY_COUNT` flaky-pool artifacts, spread across diverse repos via deterministic stride sampling (same method as the original 100-run).

**Checkpoint:** printed repo counts sum to `N_ARTIFACTS + FLAKY_COUNT`, no duplicates.

In [ ]:
"""Select N_ARTIFACTS reproducible + FLAKY_COUNT flaky artifacts across diverse repos.

Reuses the cached query results (artifacts_reproducible_py.json /
artifacts_flaky_py.json). Deterministic stride sampling (sorted by image_tag)
for within-repo spread. Writes artifacts_selection.json with fields:
image_tag, repo, failed_job_id, pool.
"""
import json
from collections import Counter

rep = json.load(open("artifacts_reproducible_py.json"))
flk = json.load(open("artifacts_flaky_py.json"))


def stride_pick(arts, k, exclude):
    cands = sorted(
        (a for a in arts if a["image_tag"] not in exclude
         and a.get("failed_job", {}).get("job_id")),
        key=lambda a: a["image_tag"],
    )
    if len(cands) < k:
        raise SystemExit(f"only {len(cands)} candidates for k={k}")
    step = len(cands) / k
    return [cands[int(i * step)] for i in range(k)]


def to_pick(a, pool):
    return {
        "image_tag": a["image_tag"],
        "repo": a["repo"],
        "failed_job_id": a["failed_job"]["job_id"],
        "pool": pool,
        "test_framework": a.get("test_framework", ""),
        "ci_service": a.get("ci_service", ""),
    }


def build_plan(pool_arts, n, max_repos):
    """Allocate n picks across repos, largest repos first. Deterministic."""
    by_repo = {}
    for a in pool_arts:
        if a.get("failed_job", {}).get("job_id"):
            by_repo.setdefault(a["repo"], []).append(a)
    ordered = sorted(by_repo, key=lambda r: -len(by_repo[r]))[:max_repos]
    plan, remaining = {}, n
    for i, r in enumerate(ordered):
        slots_left = len(ordered) - i
        give = min(len(by_repo[r]), -(-remaining // slots_left))  # ceil division
        if give > 0:
            plan[r] = give
            remaining -= give
    return plan, by_repo


picks, seen_tags = [], set()
for pool_arts, n, max_repos, pool in [
    (rep, N_ARTIFACTS, 8, "repro"),
    (flk, FLAKY_COUNT, 3, "flaky_pool"),
]:
    plan, by_repo = build_plan(pool_arts, n, max_repos)
    for repo, k in plan.items():
        for a in stride_pick(by_repo[repo], k, seen_tags):
            picks.append(to_pick(a, pool))
            seen_tags.add(a["image_tag"])

assert len({p["image_tag"] for p in picks}) == len(picks), "duplicate tags!"
json.dump(picks, open("artifacts_selection.json", "w"), indent=1)

n_repro = sum(1 for p in picks if p["pool"] == "repro")
n_flaky = sum(1 for p in picks if p["pool"] == "flaky_pool")
print(f"selected {len(picks)} artifacts ({n_repro} repro + {n_flaky} flaky_pool)")
print("repos:", len(Counter(p["repo"] for p in picks)))
for r, c in sorted(Counter(p["repo"] for p in picks).items()):
    print(f"  {r}: {c}")
print("wrote artifacts_selection.json")


selected 13 artifacts (10 repro + 3 flaky_pool)
repos: 9
  hivesolutions/appier: 1
  marshallward/f90nml: 1
  microsoft/DeepSpeed: 1
  numpy/numpy: 2
  open-telemetry/opentelemetry-python: 3
  python/mypy: 1
  terasolunaorg/guideline: 2
  wbond/package_control_channel: 1
  zarr-developers/zarr-python: 1
wrote artifacts_selection.json


## Phase 3 — Fetch failed build logs
Downloads each artifact's failed-job build log via the API. Paced at ~4 s between requests to stay under the 20 req/min unauthenticated limit, with retries.

**Checkpoint:** expect `0 empty, 0 failed` in the summary line.

In [ ]:
"""Phase 2: Fetch failed-job build logs for each selected artifact (paced, retried).

Usage: python3 phase2_fetch.py [artifacts.json] [logdir] [report.json]
"""
import json
import time
import requests
from bugswarm.common.rest_api.database_api import DatabaseAPI

artifacts_file = "artifacts_selection.json"   # written by the Select phase
logdir = "logs"
report_file = "phase2_report.json"
import os
os.makedirs(logdir, exist_ok=True)

api = DatabaseAPI()
picks = json.load(open(artifacts_file))

ok, empty, failed = 0, [], []
for i, p in enumerate(picks):
    tag = p["image_tag"]
    jid = str(p["failed_job_id"])
    out = os.path.join(logdir, f"{tag}_failed.log")
    for attempt in range(1, 4):
        try:
            log = api.get_build_log(jid)
            text = log if isinstance(log, str) else str(log)
            if text and len(text.strip()) > 100:
                open(out, "w").write(text)
                ok += 1
                print(f"[{i+1}/{len(picks)}] OK  {tag} ({len(text)} chars)", flush=True)
            else:
                empty.append(tag)
                print(f"[{i+1}/{len(picks)}] EMPTY {tag}", flush=True)
            break
        except (requests.exceptions.RequestException, ConnectionError) as e:
            print(f"[{i+1}/{len(picks)}] retry {attempt} for {tag}: {type(e).__name__}", flush=True)
            time.sleep(8 * attempt)
    else:
        failed.append(tag)
    time.sleep(4)  # stay under 20 req/min

print(f"\nDone: {ok} ok, {len(empty)} empty, {len(failed)} failed")
print("empty:", empty)
print("failed:", failed)
json.dump({"ok": ok, "empty": empty, "failed": failed},
          open(report_file, "w"))


[1/13] OK  open-telemetry-opentelemetry-python-32604074411 (113385 chars)
[2/13] OK  open-telemetry-opentelemetry-python-36077145856 (51898 chars)
[3/13] OK  terasolunaorg-guideline-101022017 (352819 chars)
[4/13] OK  terasolunaorg-guideline-165422574 (318597 chars)
[5/13] OK  numpy-numpy-100031171 (53480 chars)
[6/13] OK  python-mypy-102009774 (192436 chars)
[7/13] OK  marshallward-f90nml-118661876 (34093 chars)
[8/13] OK  wbond-package_control_channel-113007194 (30678 chars)
[9/13] OK  zarr-developers-zarr-python-24663947127 (75978 chars)
[10/13] OK  hivesolutions-appier-113213402 (61231 chars)
[11/13] OK  numpy-numpy-102015359 (309139 chars)
[12/13] OK  open-telemetry-opentelemetry-python-36077147594 (97206 chars)
[13/13] OK  microsoft-DeepSpeed-14953875047 (70105 chars)

Done: 13 ok, 0 empty, 0 failed
empty: []
failed: []


## Phase 4 — Parse logs into structured features
Extracts per log: failing test names (pytest / unittest / nose / custom formats), build stage (`install` | `test` | `build` | `other/ci-step` | `unknown`), exception type, exit code, and a pip dependency-conflict counter. All parser regexes are kept exactly as in the validated script.

**Checkpoint:** expect a parser hit rate ≥ 80% (pilot: 92%, 100-run: 95%).

In [ ]:
"""Phase 3: Parse BugSwarm failed-build logs into structured features.

Extracts per log:
  - failed_tests : list of failing test identifiers (pytest/unittest/nose/custom formats)
  - stage        : 'install' | 'test' | 'build' | 'unknown'
  - exception    : most salient *Error/*Exception name
  - exit_code    : int or None
  - failure_signal: one-line evidence snippet
Writes one JSON per artifact into parsed/.
"""
import json
import os
import re

LOGDIR = "logs"
OUTDIR = "parsed"
os.makedirs(OUTDIR, exist_ok=True)

# ---------- test-name patterns ----------
PATTERNS = [
    ("pytest_progress", re.compile(r"^.{0,40}?(\S+::\S+)\s+FAILED\b")),
    ("pytest_summary", re.compile(r"^.{0,40}?FAILED\s+(\S+::\S+|\S+)")),
    ("unittest", re.compile(r"^(?:ERROR|FAIL):\s+(\S+)(?:\s+\(([^)]+)\))?")),
    ("mypy_runner", re.compile(r"^FAILURE\s+#\d+\s+(.+)$")),
    ("tox_env", re.compile(r"^.{0,40}?\s*(\S+):\s*FAIL\s+code\s+\d+")),
    ("nose", re.compile(r"^(?:ERROR|FAIL):\s*(\S+)$")),
    # pytest collection errors: "ERROR path/to/test_x.py - <reason>" (timestamp-prefixed)
    ("pytest_collect", re.compile(r"^.{0,40}?ERROR\s+(\S+?)(?:\s+-|$)")),
]

# ---------- stage signals ----------
INSTALL_MARKERS = [
    "installing dependencies", "pip install", "pip3 install",
    "conda install", "install:", "before_install",
]
TEST_CMD_MARKERS = [
    "pytest", "nosetests", "python -m unittest", "runtests.py",
    "tox", "python setup.py test",
]
# travis fold markers for the script section
TRAVIS_SCRIPT = re.compile(r"travis_fold:start:script")

# ---------- exception ----------
# Note: the first alternative needs a CamelCase prefix (TypeError); the bare
# "Exception"/"Error" alternative covers `raise Exception(...)` tracebacks.
EXC_RE = re.compile(r"([A-Za-z_][\w.]*?(?:Error|Exception)|Exception|Error)\s*([:]|$)")
PYTEST_E_RE = re.compile(r"^E\s+([A-Za-z_][\w.]*?(?:Error|Exception|Warning))(?::|\s*$)")
WARN_RE = re.compile(r"([A-Za-z_][\w.]*?Warning)\s*[:]")
LINT_RE = re.compile(r"\b([EW]\d{3,4})\b")
# mypy type errors: "file.py:12:34: error: <msg>  [assignment]"
MYPY_CODE_RE = re.compile(r":\s*error:\s.*\[([a-z][a-z-]+)\]\s*$")
DEP_CONFLICT_RE = re.compile(r"has requirement .* incompatible", re.I)

EXIT_RES = [
    re.compile(r"exited with (\d+)"),
    re.compile(r"exit code (\d+)", re.I),
    re.compile(r"Process completed with exit code (\d+)"),
]

# ANSI color codes (break FAILED/exception regexes on GitHub Actions logs)
ANSI_RE = re.compile(r"\x1b\[[0-9;]*[a-zA-Z]|\x1b\][^\x07]*\x07")
# unified-diff output of a format/lint check step (e.g. black --check)
FORMAT_DIFF_RE = re.compile(r"^.{0,40}?\+\+\+ b/\S+")

SUMMARY_FAILED_RES = [
    re.compile(r"(\d+)\s+failed,\s+(\d+)\s+passed"),
    re.compile(r"FAILED\s*\(([^)]*)\)"),
    re.compile(r"SUMMARY\s+(\d+)/\d+\s+tasks and (\d+)/\d+ tests failed"),
    # pytest collection errors (narrow: avoid matching "mypy: Found 1 error in 1 file")
    re.compile(r"Interrupted:\s*(\d+)\s+errors?\s+during\s+collection"),
    re.compile(r"={3,}\s*(\d+)\s+errors?\s+in\s+[\d.]+s\s*={3,}"),
]


def parse(path):
    with open(path, errors="replace") as f:
        lines = [ANSI_RE.sub("", ln) for ln in f.read().splitlines()]
    text = "\n".join(lines)
    low = text.lower()

    # ---- failed tests ----
    tests = []
    for name, rx in PATTERNS:
        for ln in lines:
            m = rx.search(ln.strip())
            if m:
                t = m.group(1).strip()
                # drop summary-line noise like "(KNOWNFAIL=6,"
                if t and len(t) < 200 and not t.startswith("(") and "KNOWNFAIL" not in t and "SKIP=" not in t:
                    if t not in tests:
                        tests.append(t)
    # pytest short summary section is authoritative; keep all found

    # ---- stage ---- (computed after exception extraction; see below)
    stage = "unknown"
    first_test_line = None

    # ---- exception: priority: pytest "E ExcType" > *Error/*Exception > *Warning > lint code ----
    exc, exc_line = None, None
    for ln in reversed(lines):
        s = ln.strip()
        if len(s) > 300:
            continue
        m = PYTEST_E_RE.match(s)
        if m:
            exc, exc_line = m.group(1).split(".")[-1], s[:200]
            break
    if exc is None:
        for ln in reversed(lines):
            s = ln.strip()
            if len(s) > 300 or "copying" in s.lower():
                continue
            m = EXC_RE.search(s)
            if m:
                exc, exc_line = m.group(1).split(".")[-1], s[:200]
                break
    if exc is None:
        for ln in reversed(lines):
            s = ln.strip()
            if len(s) > 300:
                continue
            m = MYPY_CODE_RE.search(s)
            if m:
                exc, exc_line = "mypy-" + m.group(1), s[:200]
                break
    if exc is None:
        for ln in reversed(lines):
            s = ln.strip()
            if len(s) > 300:
                continue
            m = WARN_RE.search(s) or LINT_RE.search(s)
            if m:
                exc, exc_line = m.group(1).split(".")[-1], s[:200]
                break
    if exc is None:
        # format/lint check step emitting a unified diff (e.g. black --check):
        # no exception exists; the diff itself is the failure signature
        for ln in lines:
            m = FORMAT_DIFF_RE.search(ln.strip())
            if m:
                exc, exc_line = "FormatDiff", ln.strip()[:200]
                break
    if exc is None:
        # tox environment failed with no Python exception (e.g. a linter env);
        # the env name is the failing unit
        for ln in lines:
            m = re.search(r"^.{0,40}?\s*(\S+):\s*FAIL\s+code\s+\d+", ln.strip())
            if m:
                exc, exc_line = "ToxEnvFail", ln.strip()[:200]
                # make sure the env name is recorded as the failing unit
                env = m.group(1).rstrip(":")
                if env and env not in tests:
                    tests.append(env)
                break

    # A CalledProcessError is usually just the CI wrapper: prefer the root-cause
    # exception (e.g. ModuleNotFoundError) when one appears earlier in the log.
    if exc == "CalledProcessError":
        for ln in lines:
            s = ln.strip()
            if len(s) > 300:
                continue
            m = re.search(r"((?:ModuleNotFound|Import)Error)\s*:", s)
            if m:
                exc, exc_line = m.group(1), s[:200]
                break

    # ---- dependency conflicts (pip "has requirement ... incompatible") ----
    dep_conflicts = sum(1 for ln in lines if DEP_CONFLICT_RE.search(ln))

    # ---- failure evidence for non-test CI steps ----
    # nonzero exit anywhere in the log => something failed even if no tests ran
    _exit_nonzero = any(
        (m := rx.search(ln)) and int(m.group(1)) != 0
        for rx in EXIT_RES for ln in lines
    )
    failure_evidence = bool(exc) or _exit_nonzero

    # ---- stage (needs exception + test-command position) ----
    for i, ln in enumerate(lines):
        l = ln.lower()
        if any(m in l for m in TEST_CMD_MARKERS) and ("$" in ln or "[command]" in ln or ln.strip().startswith("+") or "run:" in l):
            first_test_line = i
            break
    install_fail = any(
        re.search(r"(pip|conda).*(error|failed|could not|no matching)", ln, re.I)
        for ln in lines
    ) or ("packaging backend failed" in low)
    test_fail_evidence = bool(tests) or any(
        rx.search(ln) for rx in SUMMARY_FAILED_RES for ln in lines
    )
    exc_after_test = False
    if exc and first_test_line is not None:
        for ln in lines[first_test_line:]:
            if exc in ln:
                exc_after_test = True
                break
    if install_fail and not test_fail_evidence and not exc_after_test:
        stage = "install"
    elif "packaging backend failed" in low and not exc_after_test:
        stage = "build"
    elif test_fail_evidence or exc_after_test or first_test_line is not None:
        stage = "test"
    elif failure_evidence and first_test_line is None and not install_fail:
        # A command failed (exception and/or nonzero exit) but it was never a
        # test command and not an install/packaging step: a CI diagnostic or
        # auxiliary step (e.g. ds_report crashing at import). Distinct mode.
        stage = "other/ci-step"

    # ---- exit code: last match ----
    exit_code = None
    for ln in reversed(lines):
        for rx in EXIT_RES:
            m = rx.search(ln)
            if m:
                exit_code = int(m.group(1))
                break
        if exit_code is not None:
            break

    # ---- failure signal snippet ----
    signal = None
    for ln in lines:
        s = ln.strip()
        if re.search(r"FAILED|FAILURE|errors=\d+|failed,", s) and len(s) < 220 and "copying" not in s.lower():
            signal = s[:200]
    if signal is None and exc_line:
        signal = exc_line

    return {
        "failed_tests": tests,
        "stage": stage,
        "exception": exc,
        "exit_code": exit_code,
        "failure_signal": signal,
        "dep_conflicts": dep_conflicts,
        "n_lines": len(lines),
    }


artifacts_file = "artifacts_selection.json"   # written by the Select phase
logdir = "logs"
outdir = "parsed"
os.makedirs(outdir, exist_ok=True)
picks = json.load(open(artifacts_file))
results = {}
hits = 0
for p in picks:
    tag = p["image_tag"]
    path = os.path.join(logdir, f"{tag}_failed.log")
    if not os.path.exists(path):
        results[tag] = {"error": "missing log"}
        continue
    try:
        r = parse(path)
    except Exception as e:  # noqa: BLE001
        results[tag] = {"error": f"parse crashed: {e}"}
        continue
    r["repo"] = p["repo"]
    r["pool"] = p["pool"]
    out = os.path.join(outdir, f"{tag}.json")
    json.dump(r, open(out, "w"), indent=1)
    results[tag] = r
    has_all = bool(r["failed_tests"]) and r["stage"] != "unknown" and bool(r["exception"])
    # install/build/other-ci-step failures legitimately have no test names
    if r["stage"] in ("install", "build", "other/ci-step") and r["exception"]:
        has_all = True
    hits += has_all
    print(f"{'HIT ' if has_all else 'MISS'} {tag}: tests={len(r['failed_tests'])} "
          f"stage={r['stage']} exc={r['exception']} exit={r['exit_code']}", flush=True)

n = len([p for p in picks if os.path.exists(os.path.join(logdir, f"{p['image_tag']}_failed.log"))])
rate = hits / n if n else 0
print(f"\nParser hit rate: {hits}/{n} = {rate:.0%}")
report_file = "phase3_report.json"
json.dump({"hit_rate": rate, "hits": hits, "n": n}, open(report_file, "w"))




HIT  open-telemetry-opentelemetry-python-32604074411: tests=3 stage=test exc=AssertionError exit=1
HIT  open-telemetry-opentelemetry-python-36077145856: tests=13 stage=test exc=SyntaxError exit=2
MISS terasolunaorg-guideline-101022017: tests=0 stage=unknown exc=None exit=0
MISS terasolunaorg-guideline-165422574: tests=0 stage=unknown exc=None exit=0
HIT  numpy-numpy-100031171: tests=1 stage=test exc=AssertionError exit=1
HIT  python-mypy-102009774: tests=1 stage=test exc=E126 exit=1
HIT  marshallward-f90nml-118661876: tests=0 stage=other/ci-step exc=ImportError exit=1
HIT  wbond-package_control_channel-113007194: tests=1 stage=test exc=AssertionError exit=1
HIT  zarr-developers-zarr-python-24663947127: tests=2 stage=test exc=ImportError exit=1
HIT  hivesolutions-appier-113213402: tests=6 stage=test exc=ImportError exit=1
HIT  numpy-numpy-102015359: tests=1 stage=test exc=TypeError exit=1
HIT  open-telemetry-opentelemetry-python-36077147594: tests=31 stage=test exc=SyntaxError exit=2
HI

## Phase 5 — Heuristic labeling
Rule-based first-pass labels: flaky-pool → `flaky_candidate`; test-stage → `regression`; install/build-stage → `dependency`; `other/ci-step` → `infra`; missing `ModuleNotFoundError`/`ImportError` at import time → `dependency` regardless of stage. Rows hitting any ambiguity trigger (non-test stage, network-ish exception, dep conflicts, missing tests/exception) are flagged for review — the heuristic is trustworthy for clean test-stage regressions and unreliable everywhere else.

**Checkpoint:** note the `needs_review` count — those rows feed the optional review step next.

In [ ]:
"""Phase 4a: Heuristic-label parsed artifacts; flag ambiguous cases for manual review.

Usage: python3 phase4a_heuristic.py [artifacts_100.json] [parsed_dir] [out_json]

Heuristic rules (pilot lessons applied):
  - flaky_pool                    -> flaky_candidate (always review: job-flaky != test-flaky)
  - stage == test                 -> regression
  - stage in (install, build)     -> dependency (REVIEW: phase != class, cf. falcon)
  - stage == other/ci-step        -> infra (REVIEW: env diagnostic, not code signal)
  - stage == unknown              -> unknown (REVIEW)
Review triggers: flaky pool, non-test stage, dep_conflicts>0, network-ish
exception, missing exception/tests.
"""
import json
import os
import glob

artifacts_file = "artifacts_selection.json"   # written by the Select phase
parsed_dir = "parsed"                          # written by the Parse phase
out_json = "heuristic_rows.json"

picks = {p["image_tag"]: p for p in json.load(open(artifacts_file))}

NETWORK_EXC = {
    "ConnectionError", "ConnectTimeout", "ReadTimeout", "TimeoutError",
    "SocketError", "URLError", "HTTPError", "DNSError", "SSLError",
    "NewConnectionError", "MaxRetryError",
}


def heuristic(tag, r):
    pool = r["pool"]
    stage = r["stage"]
    exc = r.get("exception")
    if pool == "flaky_pool":
        return "flaky_candidate"
    # missing module at import/collection time = dependency, regardless of stage
    if exc in ("ModuleNotFoundError", "ImportError"):
        return "dependency"
    if stage == "test":
        return "regression"
    if stage in ("install", "build"):
        return "dependency"
    if stage == "other/ci-step":
        return "infra"
    return "unknown"


rows = []
for fp in sorted(glob.glob(os.path.join(parsed_dir, "*.json"))):
    tag = os.path.basename(fp).replace(".json", "")
    r = json.load(open(fp))
    h = heuristic(tag, r)
    reasons = []
    if r["pool"] == "flaky_pool":
        reasons.append("flaky_pool")
    if r["stage"] in ("install", "build", "other/ci-step", "unknown"):
        reasons.append(f"stage={r['stage']}")
    if r.get("dep_conflicts"):
        reasons.append(f"dep_conflicts={r['dep_conflicts']}")
    if r.get("exception") in NETWORK_EXC:
        reasons.append(f"net_exc={r['exception']}")
    if not r.get("exception"):
        reasons.append("no_exception")
    if not r.get("failed_tests") and r["stage"] == "test":
        reasons.append("test_stage_no_tests")
    rows.append({
        "artifact_id": tag,
        "repo": r["repo"],
        "pool": r["pool"],
        "failed_tests": "; ".join(r["failed_tests"][:5]),
        "n_failed_tests": len(r["failed_tests"]),
        "stage": r["stage"],
        "exception": r["exception"],
        "exit_code": r["exit_code"],
        "dep_conflicts": r.get("dep_conflicts", 0),
        "heuristic_label": h,
        "needs_review": bool(reasons),
        "review_reasons": "; ".join(reasons),
        "failure_signal": (r.get("failure_signal") or "")[:220],
    })

json.dump(rows, open(out_json, "w"), indent=1)

from collections import Counter
print("heuristic distribution:", dict(Counter(x["heuristic_label"] for x in rows)))
need = [x for x in rows if x["needs_review"]]
print(f"needs_review: {len(need)}/{len(rows)}")
print("review reason counts:", dict(Counter(
    rr for x in need for rr in x["review_reasons"].split("; "))))
print(f"wrote {out_json}")


heuristic distribution: {'dependency': 3, 'flaky_candidate': 3, 'regression': 5, 'unknown': 2}
needs_review: 6/13
review reason counts: {'stage=other/ci-step': 2, 'flaky_pool': 3, 'dep_conflicts=5': 1, 'stage=unknown': 2, 'no_exception': 2}
wrote heuristic_rows.json


## Phase 6 — Manual review (OPTIONAL)
Shows every row the heuristic flagged as ambiguous. Inspect the table, and if you disagree with any label, add an entry to `OVERRIDES` below. **Leave it empty to accept all heuristic labels** — the pipeline runs end-to-end without this step.

**Checkpoint:** `OVERRIDES entries: 0` means heuristics stand.

In [ ]:
import json
import pandas as pd

rows = json.load(open("heuristic_rows.json"))
need = [r for r in rows if r["needs_review"]]
print(f"{len(need)}/{len(rows)} rows flagged for manual review "
      f"({len(rows) - len(need)} clean heuristics accepted as-is)")
if need:
    display(pd.DataFrame(need)[["artifact_id", "repo", "stage", "exception",
                                "heuristic_label", "review_reasons", "failure_signal"]])
else:
    print("nothing to review.")

# ---------------------------------------------------------------------------
# OPTIONAL overrides: fill in any rows you disagree with, then Runtime -> Run all.
# Format per artifact_id:
#   "some-tag": {"final_label": "regression",   # regression | flaky |
#                                                   # flaky_candidate | infra | dependency
#                "confidence": "high",            # high | medium | low
#                "evidence": "TypeError: ..."}    # one line quoted from the log
# Leave OVERRIDES empty to accept every heuristic label.
# ---------------------------------------------------------------------------
OVERRIDES = {
    # "numpy-numpy-106081974": {"final_label": "regression",
    #                           "confidence": "high",
    #                           "evidence": "TypeError: unsupported operand type(s)"},
}
print(f"\nOVERRIDES entries: {len(OVERRIDES)}")


6/13 rows flagged for manual review (7 clean heuristics accepted as-is)


,artifact_id,repo,stage,exception,heuristic_label,review_reasons,failure_signal
0,marshallward-f90nml-118661876,marshallward/f90nml,other/ci-step,ImportError,dependency,stage=other/ci-step,ImportError: No module named numpy
1,microsoft-DeepSpeed-14953875047,microsoft/DeepSpeed,other/ci-step,TypeError,flaky_candidate,flaky_pool; stage=other/ci-step; dep_conflicts=5,2023-07-11T17:05:10.9923932Z TypeError: Can't ...
2,numpy-numpy-102015359,numpy/numpy,test,TypeError,flaky_candidate,flaky_pool,"FAILED (KNOWNFAIL=7, SKIP=15, errors=1)"
3,open-telemetry-opentelemetry-python-36077147594,open-telemetry/opentelemetry-python,test,SyntaxError,flaky_candidate,flaky_pool,2025-01-23T17:46:09.2559762Z E SyntaxError: ...
4,terasolunaorg-guideline-101022017,terasolunaorg/guideline,unknown,None,unknown,stage=unknown; no_exception,
5,terasolunaorg-guideline-165422574,terasolunaorg/guideline,unknown,None,unknown,stage=unknown; no_exception,



OVERRIDES entries: 0


## Phase 7 — Finalize labels
Merges heuristic rows with your `OVERRIDES`. Rows without an override keep the heuristic label (clean test-stage regressions get `medium` confidence, everything unreviewed gets `low`).

**Checkpoint:** final label distribution is printed — expect it to be regression-dominated (BugSwarm structurally underrepresents flaky/infra/dependency).

In [ ]:
"""Phase 4b: Merge heuristic rows with manual overrides -> final labeled rows.

Usage: python3 phase4b_finalize.py [heuristic_rows_100.json] [manual_overrides_100.json] [out_json]

manual_overrides_100.json format:
  { "<artifact_id>": {"final_label": ..., "confidence": "high|medium|low", "evidence": "..."} }
Any row without an override keeps its heuristic label with confidence "low"
(unless it was a clean test-stage regression, which gets "medium").
"""
import json
from collections import Counter

heur_file = "heuristic_rows.json"   # written by the Heuristic-label phase
out_json = "labeled_rows.json"

rows = json.load(open(heur_file))
overrides = OVERRIDES  # defined in the Manual-review cell above (default: {} = accept heuristics)

final_rows = []
for r in rows:
    tag = r["artifact_id"]
    ov = overrides.get(tag)
    if ov:
        final, conf, ev = ov["final_label"], ov["confidence"], ov["evidence"]
        reviewed = True
    else:
        final = r["heuristic_label"]
        # unreviewed clean test-stage regressions: medium; everything else: low
        conf = "medium" if (r["heuristic_label"] == "regression"
                           and not r["needs_review"]) else "low"
        ev = r["failure_signal"]
        reviewed = False
    final_rows.append({
        "artifact_id": tag,
        "repo": r["repo"],
        "pool": r["pool"],
        "failed_tests": r["failed_tests"],
        "stage": r["stage"],
        "exception": r["exception"],
        "exit_code": r["exit_code"],
        "heuristic_label": r["heuristic_label"],
        "final_label": final,
        "confidence": conf,
        "reviewed": reviewed,
        "evidence_snippet": ev[:220],
    })

json.dump(final_rows, open(out_json, "w"), indent=1)

agree = sum(1 for x in final_rows if x["heuristic_label"] == x["final_label"])
reviewed_n = sum(1 for x in final_rows if x["reviewed"])
print("final distribution:", dict(Counter(x["final_label"] for x in final_rows)))
print("confidence:", dict(Counter(x["confidence"] for x in final_rows)))
print(f"heuristic==final agreement: {agree}/{len(final_rows)} "
      f"(over {reviewed_n} manually reviewed)")
print(f"wrote {out_json}")


final distribution: {'dependency': 3, 'flaky_candidate': 3, 'regression': 5, 'unknown': 2}
confidence: {'low': 8, 'medium': 5}
heuristic==final agreement: 13/13 (over 0 manually reviewed)
wrote labeled_rows.json


## Phase 8 — Assemble the dataset
Writes `labeled.csv` with a **project-level** train/dev/holdout split (no repo appears in more than one split — this prevents same-repo logs leaking across train/test) plus sanity checks on traceability, empty labels, and leakage.

**Checkpoint:** expect `SANITY: PASS`.

In [ ]:
"""Phase 5: Assemble labeled CSV + project-level split + sanity checks.

Usage: python3 phase5_assemble.py [rows_json] [artifacts_json] [out_csv]
       [holdout_repos_csv] [dev_repos_csv]

Split is by PROJECT (repo): no repo appears in more than one split.
Defaults reproduce the pilot (labeled_sample.csv, open-telemetry holdout).
"""
import csv
import json

from collections import Counter  # (also imported below; needed here for the split)
rows_file = "labeled_rows.json"            # written by the Finalize phase
artifacts_file = "artifacts_selection.json"  # written by the Select phase
out_csv = "labeled.csv"

rows = json.load(open(rows_file))
picks = {p["image_tag"]: p for p in json.load(open(artifacts_file))}

# Project-level split: hold out ENTIRE repos (no repo appears in two splits).
# Deterministic: smallest-count repo -> holdout, second smallest -> dev, rest -> train.
repo_counts = Counter(r["repo"] for r in rows)
ordered_repos = sorted(repo_counts, key=lambda r: (repo_counts[r], r))
holdout_repos, dev_repos = set(), set()
if len(ordered_repos) >= 3:
    holdout_repos = {ordered_repos[0]}
    dev_repos = {ordered_repos[1]}
elif len(ordered_repos) == 2:
    holdout_repos = {ordered_repos[0]}
print("holdout repos:", sorted(holdout_repos) or "none")
print("dev repos:", sorted(dev_repos) or "none")
for r in rows:
    if r["repo"] in holdout_repos:
        r["split"] = "holdout"
    elif r["repo"] in dev_repos:
        r["split"] = "dev"
    else:
        r["split"] = "train"

COLS = ["artifact_id", "repo", "split", "failed_tests", "stage", "exception",
        "exit_code", "heuristic_label", "final_label", "confidence", "evidence_snippet"]
with open(out_csv, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=COLS, extrasaction="ignore")
    w.writeheader()
    w.writerows(rows)

# ---- sanity checks ----
errors = []
for r in rows:
    if r["artifact_id"] not in picks:
        errors.append(f"{r['artifact_id']}: no traceability to {artifacts_file}")
    if not r["final_label"]:
        errors.append(f"{r['artifact_id']}: empty final_label")
    if not r["evidence_snippet"]:
        errors.append(f"{r['artifact_id']}: empty evidence")
seen = {}
for r in rows:
    seen.setdefault(r["repo"], set()).add(r["split"])
leaked = {repo: sp for repo, sp in seen.items() if len(sp) > 1}
if leaked:
    errors.append(f"LEAKAGE: repos in multiple splits: {leaked}")

from collections import Counter
n_train = sum(1 for r in rows if r["split"] == "train")
n_dev = sum(1 for r in rows if r["split"] == "dev")
n_ho = sum(1 for r in rows if r["split"] == "holdout")
print(f"rows: {len(rows)} | train: {n_train} | dev: {n_dev} | holdout: {n_ho}")
print("train repos:", sorted({r["repo"] for r in rows if r["split"] == "train"}))
print("dev repos:", sorted({r["repo"] for r in rows if r["split"] == "dev"}))
print("holdout repos:", sorted({r["repo"] for r in rows if r["split"] == "holdout"}))
print("label distribution:", dict(Counter(r["final_label"] for r in rows)))
print("SANITY:", "FAIL -> " + "; ".join(errors) if errors else "PASS")


holdout repos: ['hivesolutions/appier']
dev repos: ['marshallward/f90nml']
rows: 13 | train: 11 | dev: 1 | holdout: 1
train repos: ['microsoft/DeepSpeed', 'numpy/numpy', 'open-telemetry/opentelemetry-python', 'python/mypy', 'terasolunaorg/guideline', 'wbond/package_control_channel', 'zarr-developers/zarr-python']
dev repos: ['marshallward/f90nml']
holdout repos: ['hivesolutions/appier']
label distribution: {'dependency': 3, 'flaky_candidate': 3, 'regression': 5, 'unknown': 2}
SANITY: FAIL -> terasolunaorg-guideline-101022017: empty evidence; terasolunaorg-guideline-165422574: empty evidence


## Download the result

In [ ]:
try:
    from google.colab import files
    files.download("labeled.csv")
    print("downloaded labeled.csv")
except Exception as e:  # not on Colab: the file is already in the working dir
    import os
    print(f"colab download skipped ({type(e).__name__}); "
          f"labeled.csv is at {os.path.join(os.getcwd(), 'labeled.csv')}")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

downloaded labeled.csv
